# 06 — Ensemble Aggregation & Robustness Grid

Final notebook. Aggregates all model fits + validation results into headline tables and the full robustness grid per [methodology.md §6-§7](../docs/methodology.md).

**Inputs** (loaded from disk — runs independently of prior notebooks once `02-05` have populated their outputs):
- `data/results/{event}/{window}/{variant}/{model}/fit.pkl` — model fits (from 02)
- `data/validation/*.csv` — validation results (from 03-05)

**Outputs**: headline tables and ensemble plots (in-notebook), saved CSVs in `data/validation/final_*.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import DONOR_POOL_VARIANT
from lib.data import list_fits, load_fit, load_validation_table, save_validation_table
from lib.plotting import plot_ensemble_paths, plot_ensemble_gaps

MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
EVENTS = ['russia', 'hormuz']
WINDOWS = ['preferred', 'extended', 'narrow']
VARIANT = DONOR_POOL_VARIANT

print(f'Loading {len(list_fits())} fits from disk.')

Loading 30 fits from disk.
Last run: 2026-05-25 22:58:34


## Headline table — preferred specification only

Mean post-event gap (%) per model, plus ensemble median + IQR. This is the main result table for the thesis.

In [2]:
headline_rows = []
for event in EVENTS:
    event_gaps = {}
    for model in MODELS:
        fit = load_fit(event, 'preferred', model, variant=VARIANT)
        if fit is None:
            continue
        post = fit['gap'][fit['gap'].index >= fit['t0']]
        if len(post) == 0:
            continue
        mean_gap_pct = float(100 * (np.exp(post.mean()) - 1))
        event_gaps[model] = mean_gap_pct
        headline_rows.append({
            'event': event, 'model': model,
            'mean_post_gap_pct': mean_gap_pct,
            'rmspe_pre_log': fit['rmspe_pre'],
        })
    # Ensemble aggregate
    if event_gaps:
        arr = np.array(list(event_gaps.values()))
        headline_rows.append({
            'event': event, 'model': 'ENSEMBLE_MEDIAN',
            'mean_post_gap_pct': float(np.median(arr)),
            'rmspe_pre_log': np.nan,
            'iqr_lo': float(np.quantile(arr, 0.25)),
            'iqr_hi': float(np.quantile(arr, 0.75)),
            'n_models': len(arr),
        })

headline_df = pd.DataFrame(headline_rows)
save_validation_table(headline_df, 'final_headline')
headline_df.round(3)

,event,model,mean_post_gap_pct,rmspe_pre_log,iqr_lo,iqr_hi,n_models
0,russia,convex_scm,29.597,0.106,NaN,NaN,NaN
1,russia,ascm,13.341,0.106,NaN,NaN,NaN
2,russia,elastic_net,25.074,0.059,NaN,NaN,NaN
3,russia,xgboost,36.631,0.040,NaN,NaN,NaN
4,russia,bayesian_ridge,0.504,0.043,NaN,NaN,NaN
5,russia,ENSEMBLE_MEDIAN,25.074,NaN,13.341,29.597,5.0
6,hormuz,convex_scm,38.554,0.066,NaN,NaN,NaN
7,hormuz,ascm,42.913,0.066,NaN,NaN,NaN
8,hormuz,elastic_net,47.782,0.044,NaN,NaN,NaN
9,hormuz,xgboost,36.829,0.048,NaN,NaN,NaN


Last run: 2026-05-25 22:58:34


## Robustness grid — full appendix table

Every (event, window, model) cell. Shows whether the headline estimate is stable across pre-window choices.

In [3]:
grid_rows = []
for event in EVENTS:
    for window in WINDOWS:
        for model in MODELS:
            fit = load_fit(event, window, model, variant=VARIANT)
            if fit is None:
                continue
            post = fit['gap'][fit['gap'].index >= fit['t0']]
            if len(post) == 0:
                continue
            grid_rows.append({
                'event': event, 'window': window, 'model': model,
                'mean_gap_pct': float(100 * (np.exp(post.mean()) - 1)),
                'rmspe_pre_log': fit['rmspe_pre'],
                'n_post': len(post),
            })

grid_df = pd.DataFrame(grid_rows)
save_validation_table(grid_df, 'final_robustness_grid')
grid_df.round(3)

,event,window,model,mean_gap_pct,rmspe_pre_log,n_post
0,russia,preferred,convex_scm,29.597,0.106,151
1,russia,preferred,ascm,13.341,0.106,151
2,russia,preferred,elastic_net,25.074,0.059,151
3,russia,preferred,xgboost,36.631,0.040,151
4,russia,preferred,bayesian_ridge,0.504,0.043,151
5,russia,extended,convex_scm,35.695,0.216,151
6,russia,extended,ascm,19.671,0.216,151
7,russia,extended,elastic_net,41.131,0.140,151
8,russia,extended,xgboost,38.146,0.051,151
9,russia,extended,bayesian_ridge,-21.647,0.087,151


Last run: 2026-05-25 22:58:34


In [4]:
# Pivot for readability
pivot = grid_df.pivot_table(index=['event', 'model'], columns='window',
                             values='mean_gap_pct')
pivot = pivot.reindex(WINDOWS, axis=1)
pivot.round(2)

window                 preferred  extended  narrow
event  model                                      
hormuz ascm                42.91     38.93   42.77
       bayesian_ridge      43.61     39.56   49.69
       convex_scm          38.55     35.32   43.47
       elastic_net         47.78     39.54   42.12
       xgboost             36.83     37.43   39.15
russia ascm                13.34     19.67   24.99
       bayesian_ridge       0.50    -21.65  -11.34
       convex_scm          29.60     35.70   29.05
       elastic_net         25.07     41.13   34.46
       xgboost             36.63     38.15   35.44

Last run: 2026-05-25 22:58:34


## Ensemble visualization — preferred window

In [5]:
for event in EVENTS:
    fits = {}
    for model in MODELS:
        f = load_fit(event, 'preferred', model, variant=VARIANT)
        if f is not None:
            fits[model] = f
    if fits:
        plot_ensemble_paths(fits, title=f'{event.title()} — ensemble synthetic paths').show()
        plot_ensemble_gaps(fits, title=f'{event.title()} — ensemble gap (%)').show()

Last run: 2026-05-25 22:58:34


## Validation pass/fail summary (from 03_Validate)

In [6]:
valid_summary = load_validation_table('validation_summary')
if valid_summary is not None:
    print(valid_summary.round(4).to_string())
else:
    print('validation_summary.csv not found — run 03_Validate first.')

    event           model  wf_train_rmse  wf_val_rmse  wf_ratio  pf_mean_pct  pf_slope_yr   pf_r2  drift_contribution_pct
0  russia      convex_scm         0.1115       0.0858    0.7701       0.4285      10.0104  0.2097                  5.8060
1  russia            ascm         0.0457       0.0872    1.9094       0.1140       0.3351  0.0011                  0.1943
2  russia     elastic_net         0.0553       0.0939    1.6987       0.1738       1.1874  0.0092                  0.6887
3  russia         xgboost         0.0386       0.1349    3.4906       0.0799       3.5064  0.1801                  2.0337
4  russia  bayesian_ridge         0.0335       0.1710    5.0991       0.0932       0.1519  0.0003                  0.0881
5  hormuz      convex_scm         0.0589       0.1287    2.1848       0.2611      -7.0425  0.2535                 -1.7606
6  hormuz            ascm         0.0349       0.0872    2.4963       0.0668      -0.1337  0.0003                 -0.0334
7  hormuz     elastic_ne

## Cross-event transfer table (from 05_Cross_Event)

In [7]:
transfer = load_validation_table('cross_event_transfer')
if transfer is not None:
    print(transfer.round(3).to_string())
else:
    print('cross_event_transfer.csv not found — run 05_Cross_Event first.')

            model  russia_rmspe_pre  hormuz_independent_rmspe_pre  hormuz_transferred_rmspe_pre  hormuz_independent_post_gap_pct  hormuz_transferred_post_gap_pct  transferred_minus_independent_pct
0      convex_scm             0.106                         0.066                         0.109                           38.554                           46.960                              8.406
1            ascm             0.106                         0.066                         0.144                           42.913                           27.268                            -15.645
2     elastic_net             0.059                         0.044                         0.540                           47.782                           92.496                             44.714
3         xgboost             0.040                         0.048                         1.063                           36.829                          -58.098                            -94.927
4  bayesian_rid

## Reading the headline result

The headline thesis statement is the **ensemble median for Hormuz** with the IQR as model uncertainty. Russia's ensemble median serves as the **magnitude validation** — if it falls within the documented historical range for the Russia 2022 oil-price premium ($10-30/bbl, ~12-30%), the methodology is producing sensible magnitudes; if not, results should be discussed cautiously.

The cross-event transfer table tells the reader whether the Hormuz estimate's confidence should be widened due to regime drift between 2020-22 and 2024-26.